# 🚀 How 1.2 MB Fine-Tunes a 110M Parameter AI (LoRA in Practice)
### Parameter-Efficient Fine-Tuning with LoRA on BERT for Enterprise IT Helpdesk

In this notebook, we demonstrate **Low-Rank Adaptation (LoRA)** end-to-end:
* **Base Model:** `bert-base-uncased` (~110M parameters, ~440 MB in FP32).
* **LoRA Adapter:** Trains only the query and value attention projection matrices ($r=8$).
* **Result:** Only **~295,000 trainable parameters** (0.26% of the model) creating a lightweight **~1.2 MB adapter**!
* **Use Case:** Multi-class classification of enterprise IT support ticket intents.

In [ ]:
# 1. Install required packages
!pip install -q -U transformers datasets peft evaluate scikit-learn accelerate

In [ ]:
import os
import json
import torch
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
import evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🚀 Active device: {device}')

## 2. IT Helpdesk Dataset
We define 6 core categories for corporate IT ticketing.

In [ ]:
LABELS = [
    'PASSWORD_RESET',
    'ACCESS_ISSUE',
    'HARDWARE_FAILURE',
    'SOFTWARE_INSTALL',
    'NETWORK_LATENCY',
    'GENERAL_QUERY'
]

LABEL2ID = {lbl: i for i, lbl in enumerate(LABELS)}
ID2LABEL = {i: lbl for i, lbl in enumerate(LABELS)}

raw_data = [
    {'text': 'I forgot my Windows password and cannot log into my computer.', 'label': 'PASSWORD_RESET'},
    {'text': 'My login credentials expired this morning, how do I renew them?', 'label': 'PASSWORD_RESET'},
    {'text': 'I entered my password incorrectly three times and my corporate account is locked.', 'label': 'PASSWORD_RESET'},
    {'text': 'Need an urgent password reset for my company email address.', 'label': 'PASSWORD_RESET'},
    {'text': 'I came back from vacation and cannot remember my workstation password.', 'label': 'PASSWORD_RESET'},
    {'text': 'Received a security notice that my login has expired and needs a credential update.', 'label': 'PASSWORD_RESET'},
    {'text': 'Can you send me a temporary link to reset my account password?', 'label': 'PASSWORD_RESET'},
    {'text': 'My account appears locked in Active Directory.', 'label': 'PASSWORD_RESET'},
    {'text': 'Changed my password yesterday but the single sign-on system does not recognize it.', 'label': 'PASSWORD_RESET'},
    {'text': 'Unable to access the employee portal due to forgotten credentials.', 'label': 'PASSWORD_RESET'},
    {'text': 'Requesting domain unlock for my enterprise username.', 'label': 'PASSWORD_RESET'},
    {'text': 'Lost my password vault master key and cannot unlock credentials.', 'label': 'PASSWORD_RESET'},
    {'text': 'Lost my multi-factor authentication token required to reset my password.', 'label': 'PASSWORD_RESET'},
    {'text': 'Where do I click to recover my corporate network password?', 'label': 'PASSWORD_RESET'},
    {'text': 'My master credentials are failing on the main system login page.', 'label': 'PASSWORD_RESET'},
    {'text': 'Getting a 403 Forbidden error when trying to open the finance shared folder.', 'label': 'ACCESS_ISSUE'},
    {'text': 'Unable to connect to company VPN while working remotely.', 'label': 'ACCESS_ISSUE'},
    {'text': 'ERP says I do not have read permissions for the inventory module.', 'label': 'ACCESS_ISSUE'},
    {'text': 'Need access permissions granted for shared drive Z:.', 'label': 'ACCESS_ISSUE'},
    {'text': 'My user account is unauthorized to view reports in Power BI.', 'label': 'ACCESS_ISSUE'},
    {'text': 'Digital certificate error stating lack of authorization for portal entry.', 'label': 'ACCESS_ISSUE'},
    {'text': 'Recently transferred departments and lost access to network folders.', 'label': 'ACCESS_ISSUE'},
    {'text': 'Receiving Access Denied modal when launching internal billing tool.', 'label': 'ACCESS_ISSUE'},
    {'text': 'Cannot access Jira, system indicates my user is not in the appropriate usergroup.', 'label': 'ACCESS_ISSUE'},
    {'text': 'Requesting temporary local administrator privileges to complete scheduled task.', 'label': 'ACCESS_ISSUE'},
    {'text': 'Procurement portal is not giving me access to approve purchase orders.', 'label': 'ACCESS_ISSUE'},
    {'text': 'I lack permissions to enter the cloud staging environment.', 'label': 'ACCESS_ISSUE'},
    {'text': 'VPN disconnects immediately right after entering credentials.', 'label': 'ACCESS_ISSUE'},
    {'text': 'Sales team needs immediate access provisioning on the CRM.', 'label': 'ACCESS_ISSUE'},
    {'text': 'Not authorized to access client customer database records.', 'label': 'ACCESS_ISSUE'},
    {'text': 'My primary external monitor started flickering and went completely black.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'Laptop wont power on, no LED light even when plugged into the charger.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'Keyboard stopped responding and several keys are physically stuck.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'Wireless mouse stopped tracking cursor even after replacing batteries.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'Laptop fan is making an unusually loud buzzing sound and chassis is overheating.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'Power supply cable is frayed and causing intermittent electrical connection.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'Vertical colored lines appeared across my laptop screen.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'Second floor network printer has a paper jam and is beeping error code.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'Headset microphone is not detected by Windows during conference calls.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'USB-C docking station is not transmitting video signal to dual monitors.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'Accidentally spilled coffee over the corporate laptop keyboard.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'Biometric fingerprint reader fails to register or read finger.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'Desktop tower started smelling burnt and shut down abruptly.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'Integrated webcam fails to turn on, showing black screen on Teams.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'UPS battery backup at my desk is beeping continuously.', 'label': 'HARDWARE_FAILURE'},
    {'text': 'Please install VS Code and Git on my development machine.', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Need a license assigned for Adobe Acrobat Pro to sign legal contracts.', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Can IT install Docker Desktop so I can run local microservices?', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Need the latest Office 365 desktop suite installed on my laptop.', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Requesting installation of SAP Excel plugin for financial auditing.', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Need approval to install DBeaver database management client.', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Google Chrome browser appears corrupted and requires reinstallation.', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Requesting installation of Figma Desktop client for design work.', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Need enterprise endpoint security agent deployed on newly provisioned PC.', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Requesting installation and account activation for Slack desktop.', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Installer prompts for local admin credentials to add browser extension.', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Need an archive utility like 7-Zip installed to extract large files.', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Requesting latest stable runtime release of Python and Node.js.', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Can you install the accounting management desktop software?', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Need Oracle SQL Developer client installed for database querying.', 'label': 'SOFTWARE_INSTALL'},
    {'text': 'Conference room Wi-Fi drops connection every 5 minutes.', 'label': 'NETWORK_LATENCY'},
    {'text': 'Office wireless connection is extremely slow when downloading large files.', 'label': 'NETWORK_LATENCY'},
    {'text': 'Internal ERP system is experiencing severe latency and slowdowns today.', 'label': 'NETWORK_LATENCY'},
    {'text': 'Cannot load any external web pages, seems the entire internet connection dropped.', 'label': 'NETWORK_LATENCY'},
    {'text': 'Ethernet wall port at my desk stopped working, only Wi-Fi works.', 'label': 'NETWORK_LATENCY'},
    {'text': 'File uploads to cloud storage are taking hours due to bandwidth issues.', 'label': 'NETWORK_LATENCY'},
    {'text': 'Experiencing heavy jitter and packet loss during video conferencing calls.', 'label': 'NETWORK_LATENCY'},
    {'text': 'Ping latency to the production database server exceeds 800ms.', 'label': 'NETWORK_LATENCY'},
    {'text': 'External websites are unreachable, only internal intranet sites resolve.', 'label': 'NETWORK_LATENCY'},
    {'text': 'Constant packet loss while screen sharing presentations with clients.', 'label': 'NETWORK_LATENCY'},
    {'text': 'Third floor network access point seems unresponsive.', 'label': 'NETWORK_LATENCY'},
    {'text': 'Enterprise network connection is unstable across the entire sales floor.', 'label': 'NETWORK_LATENCY'},
    {'text': 'File transfer speeds to network attached storage capped at 20 KB/s.', 'label': 'NETWORK_LATENCY'},
    {'text': 'All large downloads fail halfway through due to connection timeout.', 'label': 'NETWORK_LATENCY'},
    {'text': 'Corporate intranet portal is not responding, request timed out.', 'label': 'NETWORK_LATENCY'},
    {'text': 'What is the standard procedure for ordering replacement computer peripherals?', 'label': 'GENERAL_QUERY'},
    {'text': 'Does IT support operate on weekends for emergency escalation?', 'label': 'GENERAL_QUERY'},
    {'text': 'Where can I review company information security compliance guidelines?', 'label': 'GENERAL_QUERY'},
    {'text': 'How do I configure automated cloud backups for local spreadsheets?', 'label': 'GENERAL_QUERY'},
    {'text': 'What is the direct phone extension for the on-duty IT helpdesk team?', 'label': 'GENERAL_QUERY'},
    {'text': 'How do I set up out-of-office automated email replies in Outlook?', 'label': 'GENERAL_QUERY'},
    {'text': 'What is the expected resolution SLA time for standard priority tickets?', 'label': 'GENERAL_QUERY'},
    {'text': 'Am I allowed to travel internationally with company-issued laptops?', 'label': 'GENERAL_QUERY'},
    {'text': 'Where is the designated drop-off station for e-waste and old batteries?', 'label': 'GENERAL_QUERY'},
    {'text': 'What is the policy regarding USB storage drives on company workstations?', 'label': 'GENERAL_QUERY'},
    {'text': 'Is there mandatory cybersecurity phishing awareness training available?', 'label': 'GENERAL_QUERY'},
    {'text': 'Who is responsible for approving new software procurement requests?', 'label': 'GENERAL_QUERY'},
    {'text': 'Where do I submit a request to update my office desk phone extension?', 'label': 'GENERAL_QUERY'},
    {'text': 'How do I change my default office printer in Windows 11 settings?', 'label': 'GENERAL_QUERY'},
    {'text': 'What is the URL for the official internal IT service desk portal?', 'label': 'GENERAL_QUERY'}
]

texts = [d['text'] for d in raw_data]
labels = [LABEL2ID[d['label']] for d in raw_data]

dataset = Dataset.from_dict({'text': texts, 'label': labels})
split = dataset.train_test_split(test_size=0.15, seed=42)
train_ds = split['train']
eval_ds = split['test']
print(f'Total samples: {len(raw_data)} (Train: {len(train_ds)}, Eval: {len(eval_ds)})')

## 3. Load BERT-Base and Configure LoRA
Observe how we reduce 110M base parameters to **~295K trainable parameters**.

In [ ]:
MODEL_NAME = 'bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)

train_tokenized = train_ds.map(tokenize, batched=True)
eval_tokenized = eval_ds.map(tokenize, batched=True)

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,                  # Decomposition rank
    lora_alpha=16,        # Scaling factor (scaling = alpha / r = 2.0)
    lora_dropout=0.1,
    target_modules=['query', 'value'],
    modules_to_save=['classifier']
)

model = get_peft_model(base_model, lora_config)
print('=' * 60)
model.print_trainable_parameters()
print('=' * 60)

## 4. Fine-Tuning with Hugging Face Trainer
Runs in under 1 minute on Colab GPU or CPU.

In [ ]:
def compute_metrics(eval_pred):
    metric = evaluate.load('accuracy')
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

OUTPUT_DIR = './bert-lora-helpdesk-checkpoint'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    save_total_limit=1,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

## 5. Save Adapter and Inspect File Size (The Real Proof)

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

adapter_path = os.path.join(OUTPUT_DIR, 'adapter_model.safetensors')
size_bytes = os.path.getsize(adapter_path)
size_mb = size_bytes / (1024 * 1024)

print('=' * 60)
print('🎉 LoRA ADAPTER SAVED!')
print(f'📁 Path: {adapter_path}')
print(f'⚖️ Exact Size: {size_mb:.2f} MB ({size_bytes:,} bytes)')
print(f'💡 While the full BERT-Base model weighs ~440 MB, this adapter is only {size_mb:.2f} MB!')
print('=' * 60)

## 6. Real-Time Inference Demonstration

In [ ]:
test_phrases = [
    'My laptop crashed and displayed a blue screen error.',
    'How can I reset my expired domain password?',
    'VPN connection keeps failing during home office.',
    'Can IT deploy Docker Desktop on my PC?',
    'Office Wi-Fi connection is extremely slow today.',
    'Where can I review corporate security compliance policies?'
]

model.eval()
print('🧪 INFERENCE EVALUATION:\n')
for text in test_phrases:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).squeeze()
        pred_idx = torch.argmax(probs).item()
        conf = probs[pred_idx].item() * 100
    
    print(f'Input: "{text}"')
    print(f'➡️ Prediction: {LABELS[pred_idx]} ({conf:.2f}% confidence)\n')